In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 벡터 DB : Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    (https://www.pinecone.io/ 에서 api key 생성 -> .env에 추가(PINECONE_API_KEY 등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102)(55조변경).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
document_list=loader.load_and_split(text_splitter=text_splitter)
len(document_list)

194

In [5]:
print(document_list[46].page_content)

1억5천만원 초과 3억원 이하

3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)

3억원 초과 5억원 이하

9,406만원 + (3억원을 초과하는 금액의 40퍼센트)

5억원 초과 10억원 이하

1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트)

10억원 초과

3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트)

제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>

[전문개정 2009. 12. 31.]

[제목개정 2014. 1. 1.]



제4절 세액의 계산 <개정 2009. 12. 31.>



제1관 세율 <개정 2009. 12. 31.>



























제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>



종합소득 과세표준

세율

1,400만원 이하

과세표준의 6퍼센트

1,400만원 초과 5,000만원 이하

84만원 + (1,400만원을 초과하는 금액의 15퍼센트)

5,000만원 초과 8,800만원 이하

624만원 + (5,000만원을 초과하는 금액의 24퍼센트)

8,800만원 초과 1억5

In [6]:
# embedding : upstage의 solar-embedding-1-large-passage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large-passage")

In [7]:
len(embedding.embed_query('소득세법'))

4096

In [8]:
%%time
#pinecone vector database 저장
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc= Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)
index_name="tax-index-table"

# 데이터를 처음 업로드할 때 
# database = PineconeVectorStore.from_documents(
#     documents = document_list,
#     embedding= embedding,
#     index_name=index_name
# )

# 업로드시 경고가 안보이려면 아나콘다 프롬프트 llm 환경에서 conda install -c conda-forge ipywidgets

# 업로드한 벡터db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 7.47 s
Wall time: 45.2 s


# 2. 답변 생성전 retrieval 확인

In [9]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrueved_docs = database.similarity_search(query, k=3) # 기본k값:4

In [14]:
print(retrueved_docs[0].page_content)

1억5천만원 초과 3억원 이하

3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)

3억원 초과 5억원 이하

9,406만원 + (3억원을 초과하는 금액의 40퍼센트)

5억원 초과 10억원 이하

1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트)

10억원 초과

3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트)

제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>

[전문개정 2009. 12. 31.]

[제목개정 2014. 1. 1.]



제4절 세액의 계산 <개정 2009. 12. 31.>



제1관 세율 <개정 2009. 12. 31.>



























제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>



종합소득 과세표준

세율

1,400만원 이하

과세표준의 6퍼센트

1,400만원 초과 5,000만원 이하

84만원 + (1,400만원을 초과하는 금액의 15퍼센트)

5,000만원 초과 8,800만원 이하

624만원 + (5,000만원을 초과하는 금액의 24퍼센트)

8,800만원 초과 1억5

In [15]:
# retrueved_docs[0].page_content
retrueved_doc = "\n\n--\n\n".join([doc.page_content for doc in retrueved_docs])

In [16]:
# query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
# retrueved_docs = database.similarity_search(query, k=3) 와 아래코드는 동일함

retruever = database.as_retriever(
    search_kwargs={"k":3}
)
retrueved_docs = retruever.invoke(query)

In [20]:
retrueved_docs[0].page_content

'1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)\n\n3억원 초과 5억원 이하\n\n9,406만원 + (3억원을 초과하는 금액의 40퍼센트)\n\n5억원 초과 10억원 이하\n\n1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트)\n\n10억원 초과\n\n3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트)\n\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\

# 3. 답변 생성

In [22]:
# gpt 4.1 mini
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system", "content":"당신은 최고의 한국 소득세법 전문가입니다."},
        {
            "role":"user",
            "content":f"""- [content]를 참고해서 사용자의 질문에 10줄 이내로 답변해 주세요
            - [content]:{retrueved_doc}
            - 질문:{query}"""
        }
    ],
    temperature=0.2
)

In [23]:
print(response.choices[0].message.content)

연봉 5천만원인 직장인의 종합소득 과세표준이 5천만원 이하 구간에 해당합니다.  
소득세 계산식은 84만원 + (5천만원 - 1,400만원) × 15%입니다.  
즉, 84만원 + 3,600만원 × 0.15 = 84만원 + 540만원 = 624만원입니다.  
따라서, 연봉 5천만원의 소득세 산출세액은 약 624만원입니다.  
여기에 각종 공제 및 세액공제 적용 시 실제 납부세액은 달라질 수 있습니다.  
기본공제, 부양가족 공제, 자녀세액공제 등을 고려해야 정확한 세액 산출이 가능합니다.  
또한, 원천징수나 연말정산 과정에서 차감되는 세액도 반영되어야 합니다.  
따라서 단순 계산 결과는 약 624만원이며, 실제 세액은 개인별 상황에 따라 다릅니다.


In [26]:
# solar pro2
from openai import OpenAI
import os
client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {"role":"system", "content":"당신은 최고의 한국 소득세법 전문가입니다."},
        {
            "role":"user",
            "content":f"""- [content]를 참고해서 사용자의 질문에 10줄 이내로 답변해 주세요
            - [content]:{retrueved_doc}
            - 질문:{query}"""
        }
    ],
    temperature=0.2
)

In [27]:
print(response.choices[0].message.content)

연봉 5천만원의 종합소득과세표준이 5,000만원일 경우, 소득세는 다음과 같이 계산됩니다:  

1. **5,000만원 초과 구간 적용**:  
   - 5,000만원 이하: 624만원 (5,000만원 × 24% - 624만원)  
   - 초과 금액(0원): 없음  
   - **산출세액 = 624만원**  

2. **공제 적용 후 최종 세액**:  
   - 근로소득공제, 자녀세액공제 등 추가 공제 가능 여부에 따라 변동될 수 있습니다.  

※ 정확한 세액은 공제 항목 및 과세표준에 따라 달라질 수 있으므로, 정확한 계산을 위해서는 세무사와 상담이 필요합니다.  

(답변: 10줄 이내 요약)  

**답변**:  
5,000만원 과세표준 시 기본 산출세액은 624만원입니다. 단, 근로소득공제, 자녀세액공제 등 적용 시 실제 납부세액은 감소할 수 있습니다. 정확한 금액은 공제 내역을 확인해야 합니다.


# 4. langchain 전달

In [28]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
# llm = ChatOpenAI(model = "gpt-4.1-mini")
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model="solar-pro2")

embedding = UpstageEmbeddings(model="solar-embedding-1-large-passage")

# 2. # 업로드한 벡터db를 가져올 때
vectorstore = PineconeVectorStore(
    embedding=embedding,# 질문을 임베딩하여 유사도 검색
    index_name=index_name
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})
# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요.
답을 모르면 모른다고 답하세요.
최대 3문장으로 간결하게 답변하세요.
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return  "\n\n---\n\n".join([doc.page_content for doc in documents])

In [29]:
# 6. RAG 체인 구성(LCEL 방식)
from langchain_core.runnables import RunnablePassthrough # {"query":"~"}=>"~"
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 cdontext와 query 변수 주입
    | llm 
    | StrOutputParser()
)
# 7. 실행
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
rag_chain.invoke(query)

'연봉 5천만원의 경우 종합소득 과세표준이 5,000만원 초과 8,800만원 이하 구간에 해당하므로 소득세는 **624만원 + (5,000만원 초과 금액의 24%)**로 계산됩니다.  \n예를 들어 과세표준이 6,000만원인 경우, **624만원 + (1,000만원 × 24%) = 864만원**입니다.  \n정확한 금액을 알려면 공제 항목(보험료, 의료비 등)을 적용한 과세표준이 필요합니다.'